# Notebook 3 — Step 5 Optimize performance (cache, partition, shuffle)
| | |
|---|---|
| **Case Study** | 6 — Smart City IoT Data Platform (6.1, 6.3 Real-time pipelines, ML systems) |
| **Roll No** | `<ROLL_NO>` |
| **Name** | `<NAME>` |

48k rows finish in under a second, so we first build a **scaled copy (×100 ≈ 4.8M rows, 400 junctions)**. Each experiment compares a baseline against an optimization. Each variant runs once as a warm-up and then 3 timed times, and the **median** is reported.

| # | Optimization | Why it helps |
|---|---|---|
| E1 | `cache()` / `persist()` | a DataFrame used by several actions is computed once instead of once per action |
| E2 | `spark.sql.shuffle.partitions` | the default 200 creates 200 tiny tasks on a laptop; matching it to the cores cuts the overhead |
| E3 | broadcast join | the small table is copied to every executor, so the big table is never shuffled |
| E4 | Parquet vs CSV, partition pruning | columnar and compressed; reads only the needed columns and folders |
| E5 | `repartition` vs `coalesce` | coalesce merges partitions without a shuffle (faster, but files can be uneven) |
| E6 | Adaptive Query Execution | Spark re-plans at runtime and merges tiny shuffle partitions automatically |

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from pyspark.sql import functions as F
from src import config
from src.spark_utils import get_spark
spark = get_spark("NB03-Optimization")
print("Spark", spark.version, "| Spark UI:", spark.sparkContext.uiWebUrl)

In [ ]:
FACTOR = 100   # lower to 50 if your laptop has less than 8 GB RAM
from src import scale_data
rows = scale_data.run(spark, FACTOR)

### Look at the plans first: what each optimization changes

In [ ]:
from src.batch_pipeline import junction_dim
facts = spark.read.parquet(config.p(scale_data.SCALED_PARQUET))
dim = junction_dim(spark, FACTOR * 10)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1); spark.conf.set("spark.sql.adaptive.enabled", "false")
print("=== WITHOUT broadcast ==="); facts.join(dim, "junction").groupBy("zone").count().explain()
print("=== WITH broadcast ===");    facts.join(F.broadcast(dim), "junction").groupBy("zone").count().explain()
spark.conf.set("spark.sql.adaptive.enabled", "true"); spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)

In [ ]:
print("Partition pruning: only the year=2017 folder is listed in PartitionFilters")
facts.filter("year = 2017").groupBy("junction").count().explain()

### Run all experiments (about 5–15 minutes on a laptop)

In [ ]:
from src import benchmark
results = benchmark.run(spark)
results[["experiment", "variant", "median_s", "speedup_vs_slowest_in_experiment", "note"]]

In [ ]:
from IPython.display import Image
Image(filename=str(config.PERF / "benchmark_results.png"))

> 📸 **Screenshot 5:** the results table and chart, and Spark UI → *Storage* tab while E1 runs (shows the cached DataFrame) and *SQL* tab (BroadcastHashJoin vs SortMergeJoin).

**How to read the results:** the explanations belong in the report; see Notebook 6 for the full analysis.

In [ ]:
spark.stop()